# Мониторинг фин. эффекта (витрина MSSQL)

Источник: `[OISUU_report].[dbo].[ВитринаСутяжность]`.

**Термины:** `Bpilot` = OISUU без Арх/Марийск; `Bam` = только Арх/Марийск.  
**Ручеёк / model** = `РезультатПроверки ∈ {0,1}`; **контроль** = `−100`.  
`ВызовМодельСутяжность` неинформативен для «модель работала».

**Финэффект:**
1. **Вариант 1** — model vs control: при соглашении избежанный ПСР (стандарт) − cost; без соглашения `psr_share×(OD×k+e_fee)`; `net = value(model) − value(control)`.
2. **Вариант 2** — только сегмент `111` (result=1 ∧ выплата=1 ∧ соглашение=1), стандарт `expected_psr − cost`.

**HTML:** `fin_effect_report_v1_rucheek.html`, `fin_effect_report_v2_cases.html`.

Ретро: 2 года до **2025-06-30**.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
for _p in (SRC, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

NOTEBOOK_DIR = PROJECT_ROOT / "monitoring" / "fin_effects"
DATA_DIR = NOTEBOOK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("PROJECT_ROOT", PROJECT_ROOT)
print("DATA_DIR", DATA_DIR)

In [ ]:
from IPython.display import display

from querulus.fin_effect.excel_monitoring import (
    RETRO_AS_OF_DEFAULT,
    VITRINA_TABLE_DEFAULT,
    default_demo_priors,
    estimate_monitoring_effect,
    extrapolate_to_year,
    format_sensitivity_table,
    format_summary_dict,
    infer_model_start,
    load_monitoring_frame,
    load_retro_priors,
    save_retro_priors,
    sensitivity_table,
)
from querulus.fin_effect.monitoring_analytics import build_variant_analytics
from querulus.fin_effect.monitoring_report import write_both_monitoring_htmls

SOURCE = "mssql"
VITRINA_TABLE = VITRINA_TABLE_DEFAULT
PRIORS_PATH = DATA_DIR / "retro_priors.json"
OD_COL = "СуммаОсновногоДолгаЗаявлено"
COST_COL = "СуммаКВыплате"

RETRO_PARQUET = Path(
    "/home/jovyan/old_home/querulus/data/processed/querulus_train_dataset.parquet"
)
LOOKBACK_YEARS = 2.0
RETRO_AS_OF = RETRO_AS_OF_DEFAULT
PRECISION = 0.49
MODEL_START = None
AS_OF = None

if not PRIORS_PATH.exists():
    save_retro_priors(default_demo_priors(), PRIORS_PATH)

try:
    df = load_monitoring_frame(source=SOURCE, table=VITRINA_TABLE)
    source_label = VITRINA_TABLE if SOURCE == "mssql" else SOURCE
except Exception as exc:
    print("SOURCE", SOURCE, "failed:", exc)
    print("fallback → synthetic")
    df = load_monitoring_frame(source="synthetic")
    source_label = "synthetic"

print("source", source_label, "shape", df.shape)
print("LOOKBACK / AS_OF", LOOKBACK_YEARS, RETRO_AS_OF)
print("MODEL_START", infer_model_start(df).date())
if "РезультатПроверки" in df.columns:
    display(df["РезультатПроверки"].value_counts(dropna=False))

In [ ]:
import pandas as pd

LOSS_CANDIDATES = ("Убыток", "LOSS_NUMBER", "LossNumber")
INCIDENT_CANDIDATES = (
    "НомерИнцидент",
    "НомерИнцидента",
    "INCIDENT_NUMBER",
)
MONEY_CANDIDATES = (
    "СуммаОсновногоДолгаЗаявлено",
    "СуммаОсновногоДолгаКВыплате",
    "Иные затраты",
    "СуммаКВыплате",
    "СуммаПлатежа",
    "Cумма выплаты по претензии",
    "Сумма выплат по ФУ",
    "Сумма выплаты по суду",
)

loss_col = next((c for c in LOSS_CANDIDATES if c in df.columns), None)
incident_col = next((c for c in INCIDENT_CANDIDATES if c in df.columns), None)
money_cols = [c for c in MONEY_CANDIDATES if c in df.columns]

if loss_col is None:
    raise KeyError(
        "Не найдена колонка убытка. "
        f"Проверены варианты: {LOSS_CANDIDATES}"
    )

work = df.copy()
for col in money_cols:
    work[col] = pd.to_numeric(work[col], errors="coerce")

rows_per_loss = work.groupby(loss_col, dropna=False).size()
duplicate_losses = rows_per_loss[rows_per_loss > 1]
extra_rows = int((rows_per_loss - 1).clip(lower=0).sum())
exact_duplicate_rows = int(work.duplicated(keep=False).sum())

print("=== ОБЩАЯ ПРОВЕРКА ===")
print(f"Строк в витрине: {len(work):,}")
print(
    "Уникальных убытков: "
    f"{work[loss_col].nunique(dropna=False):,}"
)
print(
    "Убытков с несколькими строками: "
    f"{len(duplicate_losses):,}"
)
print(f"Лишних строк относительно 1 строки/убыток: {extra_rows:,}")
print(f"Строк, входящих в полные дубли: {exact_duplicate_rows:,}")

if not duplicate_losses.empty:
    print("\nРаспределение числа строк на дублирующийся убыток:")
    print(duplicate_losses.value_counts().sort_index().to_string())

if incident_col is not None:
    losses_per_incident = work.groupby(
        incident_col,
        dropna=False,
    )[loss_col].nunique(dropna=False)
    print("\n=== УРОВЕНЬ ИНЦИДЕНТА ===")
    print(
        "Уникальных инцидентов: "
        f"{work[incident_col].nunique(dropna=False):,}"
    )
    print(
        "Инцидентов с несколькими убытками: "
        f"{int((losses_per_incident > 1).sum()):,}"
    )

print("\n=== ПОВТОРЕНИЕ ДЕНЕЖНЫХ ЗНАЧЕНИЙ ===")
money_diagnostics = []
for col in money_cols:
    raw_sum = float(work[col].fillna(0).sum())
    unique_loss_value_sum = float(
        work[[loss_col, col]]
        .drop_duplicates()[col]
        .fillna(0)
        .sum()
    )
    repeated_pairs = int(
        work.groupby([loss_col, col], dropna=False)
        .size()
        .gt(1)
        .sum()
    )
    varying_losses = int(
        work.groupby(loss_col, dropna=False)[col]
        .nunique(dropna=False)
        .gt(1)
        .sum()
    )
    money_diagnostics.append(
        {
            "column": col,
            "raw_sum": raw_sum,
            "sum_unique_loss_value": unique_loss_value_sum,
            "possible_inflation": raw_sum - unique_loss_value_sum,
            "repeated_loss_value_pairs": repeated_pairs,
            "losses_with_different_values": varying_losses,
        }
    )

display(pd.DataFrame(money_diagnostics))

print("\n=== ИТОГ ===")
if duplicate_losses.empty:
    print("Дубли по номеру убытка не обнаружены.")
else:
    print(
        "Дубли по номеру убытка обнаружены. Нужно определить, "
        "являются ли они результатом JOIN или реальными документными строками."
    )

In [ ]:
import pandas as pd

from querulus.fin_effect.excel_explore import (
    _to_numeric,
    resolve_column,
    resolve_model_payout_loss_column,
)

# Таблица должна считаться только на реальной MSSQL-витрине.
if source_label == "synthetic":
    raise RuntimeError(
        "Таблица комбинаций не строится на synthetic fallback. "
        "Проверьте подключение к MSSQL и перезапустите ячейку загрузки df."
    )

result_col = resolve_column(df, "result_check")
payout_col = resolve_model_payout_loss_column(df)
agreement_col = resolve_column(df, "agreement")
if result_col is None or payout_col is None or agreement_col is None:
    raise KeyError(
        "Нужны колонки РезультатПроверки, Выплата по модели "
        "и Заключено соглашение"
    )

combo_source = pd.DataFrame(
    {
        "Результат": _to_numeric(df[result_col]),
        "Выплата": _to_numeric(df[payout_col]).fillna(0).eq(1).astype(int),
        "Соглашение": _to_numeric(df[agreement_col]).fillna(0).eq(1).astype(int),
    },
    index=df.index,
)

# Полный перебор: 3 результата × 2 выплаты × 2 соглашения = 12 строк.
combo_grid = pd.MultiIndex.from_product(
    [[0, 1, -100], [0, 1], [0, 1]],
    names=["Результат", "Выплата", "Соглашение"],
).to_frame(index=False)
combo_counts = (
    combo_source.groupby(
        ["Результат", "Выплата", "Соглашение"],
        dropna=False,
    )
    .size()
    .rename("Количество")
    .reset_index()
)
result_payout_agreement = combo_grid.merge(
    combo_counts,
    on=["Результат", "Выплата", "Соглашение"],
    how="left",
)
result_payout_agreement["Количество"] = (
    result_payout_agreement["Количество"].fillna(0).astype(int)
)
result_payout_agreement["Доля от витрины, %"] = (
    result_payout_agreement["Количество"].div(len(df)).mul(100).round(2)
)

display(result_payout_agreement)

In [ ]:
import pandas as pd
from querulus.fin_effect.excel_monitoring import RetroPriors, compute_retro_priors

if RETRO_PARQUET is not None and Path(RETRO_PARQUET).exists():
    retro = pd.read_parquet(RETRO_PARQUET)
    priors = compute_retro_priors(
        retro,
        precision=PRECISION,
        lookback_years=LOOKBACK_YEARS,
        as_of=RETRO_AS_OF,
    )
    save_retro_priors(priors, PRIORS_PATH)
    print("priors from parquet", priors.window_start, "…", priors.window_end)
else:
    priors = load_retro_priors(PRIORS_PATH)
    if PRECISION is not None:
        priors = RetroPriors(
            precision=float(PRECISION),
            k=priors.k,
            p_pret=priors.p_pret,
            p_fu=priors.p_fu,
            p_court=priors.p_court,
            fu_fee=priors.fu_fee,
            court_fee=priors.court_fee,
            lookback_years=priors.lookback_years,
            date_column=priors.date_column,
            window_start=priors.window_start,
            window_end=priors.window_end,
            n_rows=priors.n_rows,
            n_pos=priors.n_pos,
            psr_share=priors.psr_share,
        )
        save_retro_priors(priors, PRIORS_PATH)
    print("priors from json")

print(priors)
print("e_fee", round(priors.expected_fee(), 2))

In [ ]:
effect_v1 = estimate_monitoring_effect(
    df, priors, od_col=OD_COL, cost_col=COST_COL, variant=1
)
effect_v2 = estimate_monitoring_effect(
    df, priors, od_col=OD_COL, cost_col=COST_COL, variant=2
)

print("=== variant 1 (model vs control by agreement) ===")
display(format_summary_dict({
    "model_value": effect_v1.details.get("model_value"),
    "control_value": effect_v1.details.get("control_value"),
    "net": effect_v1.net,
    "net_per_case": effect_v1.details.get("net_per_case"),
    "model_avoided": effect_v1.details.get("model_avoided"),
    "model_open_psr": effect_v1.details.get("model_open_psr"),
    "model_cost": effect_v1.details.get("model_cost"),
    "control_open_psr": effect_v1.details.get("control_open_psr"),
}))
print("=== variant 2 (segment 111 only) ===")
display(format_summary_dict({
    "n_111": effect_v2.n_intervention,
    "expected_psr": effect_v2.expected_psr,
    "cost": effect_v2.cost,
    "net": effect_v2.net,
}))

display(format_sensitivity_table(
    sensitivity_table(df, priors, od_col=OD_COL, cost_col=COST_COL, variant=1)
))

annual_v1 = extrapolate_to_year(
    effect_v1, df, model_start=MODEL_START, as_of=AS_OF, year_days=365.0
)
annual_v2 = extrapolate_to_year(
    effect_v2, df, model_start=MODEL_START, as_of=AS_OF, year_days=365.0
)

analytics_v1 = build_variant_analytics(df, amount_col=COST_COL, variant=1)
analytics_v2 = build_variant_analytics(df, amount_col=COST_COL, variant=2)

print("segments v1")
display(analytics_v1["segments_bpilot"])
print("segments v2")
display(analytics_v2["segments_bpilot"])

p1, p2 = write_both_monitoring_htmls(
    effect_v1,
    effect_v2,
    DATA_DIR,
    annual_v1=format_summary_dict(annual_v1),
    annual_v2=format_summary_dict(annual_v2),
    analytics_v1=analytics_v1,
    analytics_v2=analytics_v2,
    source_label=str(source_label),
)
print("HTML v1 →", p1)
print("HTML v2 →", p2)